# Group 5: Exercises

This notebook contains solutions to exercise 5 from Chapter 5 of "An Introduction to Statistical Learning" (Python edition).

In [1]:
import numpy as np
import statsmodels.api as sm
from ISLP import confusion_table, load_data
from ISLP.models import ModelSpec as MS
from ISLP.models import summarize
from sklearn.model_selection import train_test_split

In [2]:
default = load_data('Default')
default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [3]:
default.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   default  10000 non-null  category
 1   student  10000 non-null  category
 2   balance  10000 non-null  float64 
 3   income   10000 non-null  float64 
dtypes: category(2), float64(2)
memory usage: 176.2 KB


In [4]:
np.unique(default['default'], return_counts=True)

(array(['No', 'Yes'], dtype=object), array([9667,  333]))

**(a)**

In [5]:
design = MS(['income', 'balance'])

X = design.fit_transform(default)
y = default['default'] == 'Yes'

glm = sm.GLM(y, 
            X,
            family=sm.families.Binomial())

results = glm.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-11.540500,0.435000,-26.544,0.0
income,0.000021,0.000005,4.174,0.0
balance,0.005600,0.000000,24.835,0.0


**(b)**

In [6]:
design = MS(['income', 'balance'])

X = design.fit_transform(default)
y = default['default'] == 'Yes'

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=1)
lr = sm.GLM(y_train, 
            X_train,
            family=sm.families.Binomial())

results = lr.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-11.858100,0.528000,-22.477,0.0
income,0.000022,0.000006,3.707,0.0
balance,0.005900,0.000000,21.078,0.0


In [7]:
pred_proba = results.predict(X_valid)
pred_proba.head()

9953    0.000919
3850    0.008410
4962    0.000806
3886    0.003090
5437    0.092877
dtype: float64

In [8]:
pred = np.where(pred_proba > 0.5, 1, 0)
np.unique(pred, return_counts=True)

(array([0, 1]), array([2951,   49]))

In [9]:
conf_mat = confusion_table(pred, y_valid)
conf_mat

Truth,False,True
Predicted,,
False,2893,58
True,16,33


In [10]:
(58+ 16)/conf_mat.sum().sum()

np.float64(0.024666666666666667)

We can see that the validation set error is $2.47\%$.

(c) Repeating the process above with 3 different splits:

In [11]:
# helper function that fits the same model above but takes a split_random_state to make a different split of the data
# returns the validation set error
def fit_and_test(split_random_state):

    design = MS(['income', 'balance'])

    X = design.fit_transform(default)
    y = default['default'] == 'Yes'

    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=split_random_state)
    results = sm.GLM(y_train, 
                X_train,
                family=sm.families.Binomial()).fit()
    
    pred_proba = results.predict(X_valid)
    pred = np.where(pred_proba > 0.5, 1, 0)
    conf_mat = confusion_table(pred, y_valid)
    return (conf_mat.iloc[0, 1]+ conf_mat.iloc[1, 0])/conf_mat.sum().sum()

In [12]:
fit_and_test(1)

np.float64(0.024666666666666667)

This is the result using the same split used in **(b)**.

In [13]:
fit_and_test(2)

np.float64(0.023666666666666666)

In [14]:
fit_and_test(3)

np.float64(0.025)

In [15]:
fit_and_test(4)

np.float64(0.025333333333333333)

Looking at the validation test errors returned by the 3 different splits, we can see that there's a slight variation between them.

**(d)**

In [16]:
design = MS(['income', 'balance', 'student'])

X = design.fit_transform(default)
y = default['default'] == 'Yes'

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=1)
lr = sm.GLM(y_train, 
            X_train,
            family=sm.families.Binomial())

results = lr.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-11.142800,0.59400,-18.743,0.000
income,0.000004,0.00001,0.367,0.713
balance,0.005900,0.00000,21.022,0.000
student[Yes],-0.683600,0.28000,-2.440,0.015


In [17]:
pred_proba = results.predict(X_valid)
pred = np.where(pred_proba > 0.5, 1, 0)
conf_mat = confusion_table(pred, y_valid)
conf_mat

Truth,False,True
Predicted,,
False,2895,59
True,14,32


In [18]:
(conf_mat.iloc[0, 1]+ conf_mat.iloc[1, 0])/conf_mat.sum().sum()

np.float64(0.024333333333333332)

Including the dummy variable for `student` led to a very miniscule reduction in the test error rate from $2.47\%$ to $2.43\%$ though that could be a result of the variance, it also made the `income` variable non-statistically significant.